In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(msigdbr)
    library(viridis)
    library(glmmTMB)
    library(ggeffects)
    library(orthogene)
    library(data.table)
})

In [ ]:
estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
  cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
  aggregate_g1s_expression = counts(cds_g1s)
  aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
  aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)
  
  cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
  aggregate_g2m_expression = counts(cds_g2m)
  aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
  aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
  cds$g1s_score = log(aggregate_g1s_expression+1)
  cds$g2m_score = log(aggregate_g2m_expression+1)
  cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
  return(cds)
}

aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
  cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
  aggregate_marker_expression = counts(cds_gene_list_norm)
  aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
  aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
  aggregate_marker_expression = log(aggregate_marker_expression + 1)
  colData(cds)[[name]] = aggregate_marker_expression
  return(cds)
}


In [ ]:
df_cell = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas/df_cell.RDS")

In [ ]:
df_gene = readRDS("/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas/df_gene.RDS")

In [ ]:
rownames(df_cell) = df_cell$sample
rownames(df_gene) = df_gene$gene_id

In [ ]:
df_cell = df_cell %>%
    filter(RT_group != "293T_3T3") %>%
    mutate(Size_Factor = All_reads / exp(mean(log(All_reads))))

In [ ]:
dim(df_cell)

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
model_glmm_0.7_0.4 = readRDS("../data/nucleolar_model_glmm_Z5_0.7_0.4_n_6.RDS")

res_df_0.7_0.4 = readRDS("../data/nucleolar_genes_Z5_0.7_0.4_n_6.RDS")

In [ ]:
res_df_0.7_0.4 = res_df_0.7_0.4 %>%
    mutate(gene_short_name = case_match(as.character(gene_short_name),
        "AI314180" ~ "Ecpas",
        .default = gene_short_name
))

In [ ]:
gene_df_0.7_0.4 <- orthogene::convert_orthologs(gene_df = as.character(res_df_0.7_0.4$gene_short_name), 
                                        method = "babelgene",
                                        as_sparse = F,
                                        gene_input = "gene_ID", 
                                        gene_output = "columns", 
                                        input_species = "mouse",
                                        output_species = "human",
                                        non121_strategy = "keep_both_species") 

In [ ]:
res_df_0.7_0.4 %>%
    mutate(input_gene = as.character(gene_short_name)) %>%
    select(input_gene, Spearman) %>%
    left_join(., gene_df_0.7_0.4)

In [ ]:
manual_mapping_df <- data.frame(
    input_gene = c("Snhg4", "Erdr1"), 
    ortholog_gene = c("SNHG4", "ERDR1"),
    row.names = c("Snhg4", "Erdr1")
)

gene_df_0.7_0.4 = bind_rows(gene_df_0.7_0.4, manual_mapping_df)
gene_df_0.7_0.4

In [ ]:
gene_set_BP = msigdbr::msigdbr(species = "human", subcategory = "GO:BP")
gene_set_hallmarks = msigdbr::msigdbr(species = "human", category = "H")

gene_set_df_BP = gene_set_BP %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_hallmarks_df = gene_set_hallmarks %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

granule_genes = gene_set_df_BP %>%
    filter(gs_name == "GOBP_STRESS_GRANULE_ASSEMBLY")
atf6_genes = gene_set_df_BP %>%
    filter(grepl("GOBP_ATF6", gs_name)) %>%
    pull(gene_short_name)
ire1_genes = gene_set_df_BP %>%
    filter(grepl("GOBP_IRE1", gs_name)) %>%
    pull(gene_short_name)
perk_genes = gene_set_df_BP %>%
    filter(grepl("GOBP_PERK", gs_name)) %>%
    pull(gene_short_name)
upr_genes = gene_set_hallmarks_df %>%
    filter(grepl("UNFOLDED", gs_name)) %>%
    pull(gene_short_name)

In [ ]:
list.files(path = '/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas', pattern = 'cds_*', full.names = T)[-6]

In [ ]:
cds_names = list.files(path = '/net/trapnell/vol1/home/khj3017/proj/vcs-rna-seq/human-fetal-atlas', pattern = 'cds_*', full.names = T)[-6]

df_all = NULL

for (cds_name in cds_names) {
    
    cds = readRDS(cds_name)
    cds = cluster_cells(cds, resolution = 1e-4)
    cds$Cluster = clusters(cds)
    
    
    cds = estimate_cell_cycle(cds, 
                               g1s_markers = cc.genes$s.genes, 
                               g2m_markers = cc.genes$g2m.genes)
    cds = aggregate_gene_score(cds, granule_genes$gene_short_name, "granule_score")
    cds = aggregate_gene_score(cds, atf6_genes, "atf6_upr_score")
    cds = aggregate_gene_score(cds, ire1_genes, "ire1_upr_score")
    cds = aggregate_gene_score(cds, perk_genes, "perk_upr_score")
    cds = aggregate_gene_score(cds, upr_genes, "upr_score")
    
    cds = aggregate_gene_score(cds, gene_df_0.7_0.4$ortholog_gene, 
                               "nuc_score_0.7_0.4")

    
    metadata = as.data.frame(colData(cds))
    
    umap_coordinates = reducedDims(cds)[['UMAP']]
    
    metadata = metadata %>%
                 mutate(UMAP1 = umap_coordinates[,1],
                        UMAP2 = umap_coordinates[,2], 
                        UMAP3 = umap_coordinates[,3])
    
    if (is.null(df_all)) {
        df_all = metadata
    } else {
        df_all = rbind(df_all, metadata)
    }

    print(paste0(cds_name, " done"))
    flush.console()
    gc()
    
}

In [ ]:
s_name <- "0.7_0.4"

normalize <- function(x) { (x - min(x, na.rm=TRUE)) / (max(x, na.rm=TRUE) - min(x, na.rm=TRUE)) }

model_file <- paste0("nucleolar_model_glmm_Z5_", s_name, "_n_6.RDS")
raw_score_col  <- paste0("nuc_score_", s_name) 
norm_score_col <- paste0("normalized_nuc_score_", s_name)
pred_col       <- paste0("score_pred_glmm_", s_name)

temp_model <- readRDS(model_file)

df_all[[norm_score_col]] <- normalize(df_all[[raw_score_col]])

target_var <- all.vars(formula(temp_model))[2] 

pred_df <- data.frame(df_all[[norm_score_col]])
colnames(pred_df) <- target_var

df_all[[pred_col]] <- as.double(predict(temp_model, 
                                        newdata = pred_df, 
                                        re.form = NA, 
                                        type = "response"))
        

In [ ]:
df_all = df_all %>%
    mutate(celltype = ifelse(is.na(Organ_cell_lineage), "Unknown",
        str_split(Organ_cell_lineage, "-", simplify = T)[, 2]))

In [ ]:
dim(df_all)
head(df_all)

In [ ]:
saveRDS(df_all, "human_fetal_atlas_df_glmm_embryo.RDS")